In [6]:
import os
import pandas as pd
import joblib
import shap
import matplotlib.pyplot as plt
import gradio as gr
from langchain_groq import ChatGroq
from langchain.tools import tool
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage
from pathlib import Path
import gc
import sys
import os

#Indicamos el workspace principal (dos niveles atrás)
sys.path.append(os.path.abspath('../'))

from src.utils_pisa import *


%load_ext autoreload
%autoreload 2


establecer_semilla(42)
pd.set_option('display.float_format', lambda valor: f'{valor:.2f}')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✅ Semillas del proyecto establecidas con éxito (seed=42).


In [ ]:

ruta_datos_llm = Path('../agente_llm/data_llm')
# 1. Configuración de Groq

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

# Cargamos los 4 archivos de la imagen
dataset_hibrido = joblib.load(ruta_datos_llm/'dataset_pisa_estrato_llm.pkl')
modelo_xgb = joblib.load(ruta_datos_llm/"modelo_xgb_math_opt.pkl")
columnas_esperadas = joblib.load(ruta_datos_llm/"columnas_modelo.pkl")
explainer = joblib.load(ruta_datos_llm/"shap_explainer.pkl")

In [22]:


# 3. Herramienta: Explicar y Graficar SHAP
@tool
def graficar_relacion_variables(pais: str, variable_principal: str, variable_secundaria: str) -> str:
    """
    Genera un gráfico SHAP para analizar cómo impacta una variable en la nota y su interacción con otra.
    - pais: El código de 3 letras del país (ej. 'ESP' para España, 'USA' para Estados Unidos). Si el usuario dice "global", pasa "TODOS".
    - variable_principal: El nombre exacto de la columna principal a analizar (ej. 'tiempo_deberes_totales_diarios').
    - variable_secundaria: El nombre exacto de la columna para el color de interacción (ej. 'Gasto_Alumno_PPP').
    """
    # 1. Filtrar por país si no es una consulta global
    if pais != "TODOS":
        fila_datos = dataset_hibrido[dataset_hibrido['CNT'] == pais]
        if fila_datos.empty:
            return f"No hay datos para el país {pais}."
    else:
        fila_datos = dataset_hibrido

    # 2. Traducción al One-Hot Encoding al vuelo
    df_dummies = pd.get_dummies(fila_datos)
    df_final_xgboost = df_dummies.reindex(columns=columnas_esperadas, fill_value=0)
    
    # 3. Calcular los valores SHAP para el país filtrado
    shap_values = explainer.shap_values(df_final_xgboost)
    
    # 4. Generar el gráfico de Dependencia SHAP
    plt.figure(figsize=(8, 6))
    
    try:
        shap.dependence_plot(
            variable_principal, 
            shap_values, 
            df_final_xgboost, 
            interaction_index=variable_secundaria,
            show=False # Evita que se imprima directamente en consola, permitiendo guardarlo
        )
        ruta_imagen = "shap_relacion.png"
        plt.savefig(ruta_imagen, bbox_inches='tight')
        plt.close()
        
        return (
            f"Gráfico de relación generado. Explícale al usuario la tendencia que el modelo ha "
            f"encontrado entre {variable_principal} y {variable_secundaria} para {pais}, y MUESTRA "
            f"LA IMAGEN CON ESTE CÓDIGO MARKDOWN: ![Gráfico SHAP Relación]({ruta_imagen})"
        )
    except Exception as e:
        plt.close()
        return f"Error al generar el gráfico. Asegúrate de que las variables existen. Detalle: {str(e)}"

# 4. Ensamblaje del Agente LangGraph
herramientas = [graficar_relacion_variables]
agente_pisa = create_react_agent(llm, herramientas)

# 5. Lógica de Interfaz Gradio
def responder_chat(mensaje, historial):
    estado_inicial = {"messages": [HumanMessage(content=mensaje)]}
    resultado = agente_pisa.invoke(estado_inicial)
    
    # Extraemos la respuesta textual final del agente
    respuesta_final = resultado["messages"][-1].content
    return respuesta_final

# 6. Lanzar la interfaz incrustada en Jupyter
interfaz = gr.ChatInterface(
    fn=responder_chat,
    title="Asistente PISA - Inferencias y SHAP",
    description="Pídele al agente que te explique un estrato (ej. 'Analiza el impacto SHAP para el estrato USA08')."
)

# inline=True hace que el chat aparezca debajo de esta misma celda
interfaz.launch(inbrowser=True,inline=False)

C:\Users\victo\AppData\Local\Temp\ipykernel_25800\580536495.py:51: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agente_pisa = create_react_agent(llm, herramientas)


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
